In [113]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [114]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('smart-mcq-solver-challenge')

print("Path to competition files:", path)

Path to competition files: /kaggle/input/competitions/smart-mcq-solver-challenge


In [115]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from lightgbm import LGBMClassifier

from scipy.sparse import hstack

In [116]:
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

train.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [117]:
print(train.columns)
print(test.columns)

Index(['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer'], dtype='object')
Index(['id', 'prompt', 'A', 'B', 'C', 'D', 'E'], dtype='object')


In [118]:
def combine(df):
    return (
        "Question: " + df["prompt"].fillna("") +
        " A: " + df["A"].fillna("") +
        " B: " + df["B"].fillna("") +
        " C: " + df["C"].fillna("") +
        " D: " + df["D"].fillna("") +
        " E: " + df["E"].fillna("")
    )

train_text = combine(train)
test_text = combine(test)

In [119]:
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1,2),
    stop_words="english"
)

X_train = tfidf.fit_transform(train_text)
X_test = tfidf.transform(test_text)

In [120]:
le = LabelEncoder()

y = le.fit_transform(train["answer"])

In [121]:
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

NameError: name 'X' is not defined

In [ ]:
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

model = LGBMClassifier(
    objective="multiclass",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=63,
    random_state=42
)

model.fit(
    X_train,
    y_train,

    eval_set=[(X_valid, y_valid)],

    callbacks=[
        early_stopping(30),
        log_evaluation(50)
    ]
)

print(model.best_iteration_)

In [ ]:
probs = model.predict_proba(X_test)

probs.shape

In [ ]:
labels = le.classes_

predictions = []

for row in probs:
    idx = np.argsort(row)[::-1][:3]
    predictions.append(" ".join(labels[idx]))

predictions[:5]

In [ ]:
submission = pd.DataFrame({
    "ID": test["id"],
    "Prediction": predictions
})

submission.to_csv("/kaggle/working/submission.csv", index=False)

submission.head()